# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR² Mental Health Survey dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset source is defined by a Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json)

This dataset contains survey responses on mental health indicators, demographics, and standardized assessment scores collected from residents of Kilifi County.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
# For visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Define the dataset url
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Published: {metadata.datePublished}")
print(f"License: {metadata.license}")
print(f"Data collection timeframe: {getattr(metadata, 'dataCollectionTimeframe', 'N/A')}")

## 2. Data Overview
Explore the available record sets and their fields by referencing their `@id`. This enables precise access to dataset entities for subsequent processing.


In [ ]:
# Get available record sets and fields by their @id
record_sets = dataset.record_sets()
print("Available record sets:")
for record_set in record_sets:
    print(f"- RecordSet name: {record_set.name}, @id: {record_set.id}")
    fields = record_set.fields
    print("  Fields:")
    for field in fields:
        print(f"    - Field name: {field.name}, @id: {field.id}, dataType: {field.dataType}")
    print()

# For reference: collect all record set IDs and choose one for detailed extraction
record_set_ids = [record_set.id for record_set in record_sets]
if record_set_ids:
    example_record_set_id = record_set_ids[0]
    print(f"Example record set @id: {example_record_set_id}")
else:
    example_record_set_id = None


## 3. Data Extraction
Load data from each record set into DataFrames for analysis. Use the record set and field `@id`s identified in the overview above.

In [ ]:
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Record set {record_set_id} loaded: shape {df.shape}, columns: {df.columns.tolist()}")

# Display the first few rows from the primary record set
if example_record_set_id:
    print(f"\nHead of DataFrame for record set {example_record_set_id}:")
    display(dataframes[example_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's apply basic processing on numeric fields and demonstrate filtering, normalization, and grouping.
- We select a numeric field (by `@id`) for analysis.
- Filter records based on a threshold.
- Normalize values.
- Group by a demographic or categorical attribute.

In [ ]:
# Identify a numeric field and a group field from the record set's fields
numeric_field_id = None
group_field_id = None

if example_record_set_id:
    # Loop through fields for this record set
    for record_set in record_sets:
        if record_set.id == example_record_set_id:
            for field in record_set.fields:
                # Choose a field with dataType Float or Integer
                if field.dataType in ["schema:Float", "schema:Integer"] and numeric_field_id is None:
                    numeric_field_id = field.id
                # Choose categorical/demographic field for grouping
                if field.dataType == "schema:Text" and group_field_id is None and ("gender" in field.name.lower() or "village" in field.name.lower() or "level_of_education" in field.name.lower()):
                    group_field_id = field.id

    selected_df = dataframes[example_record_set_id]
    # If no numeric and group fields found, use column names as fallback
    if numeric_field_id is None and selected_df.shape[1] > 0:
        for col in selected_df.columns:
            if selected_df[col].dtype in ["int64", "float64"]:
                numeric_field_id = col
                break
    if group_field_id is None:
        possible_groups = [col for col in selected_df.columns if "village" in col.lower() or "gender" in col.lower() or "level_of_education" in col.lower()]
        if possible_groups:
            group_field_id = possible_groups[0]

    # Print selected fields
    print(f"Selected numeric field for analysis: {numeric_field_id}")
    print(f"Selected group field for analysis: {group_field_id}")

    # EDA: Filter, normalize, group
    if numeric_field_id in selected_df.columns:
        threshold = selected_df[numeric_field_id].quantile(0.75) if pd.api.types.is_numeric_dtype(selected_df[numeric_field_id]) else 10
        filtered_df = selected_df[selected_df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize
        mean_val = filtered_df[numeric_field_id].mean()
        std_val = filtered_df[numeric_field_id].std()
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean_val) / std_val
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by group_field if present
        if group_field_id in selected_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head())

## 5. Visualization
Visualize data distributions and relationships. Examples include histograms or grouped bar charts using the selected fields.

In [ ]:
# Draw histogram of the numeric field
if example_record_set_id and numeric_field_id in dataframes[example_record_set_id].columns:
    plt.figure(figsize=(6, 4))
    sns.histplot(dataframes[example_record_set_id][numeric_field_id], bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

# Draw mean values grouped by a categorical field
if example_record_set_id and numeric_field_id in dataframes[example_record_set_id].columns and group_field_id in dataframes[example_record_set_id].columns:
    grouped = dataframes[example_record_set_id].groupby(group_field_id)[numeric_field_id].mean().reset_index()
    plt.figure(figsize=(8, 4))
    sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped)
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
This notebook illustrated how to use `mlcroissant` to load and analyze a Croissant-structured dataset. Through exploration, we:
- Loaded metadata and record sets using the Croissant schema.
- Referenced entities by their `@id` throughout for reproducibility.
- Conducted basic EDA including filtering, normalization, and grouping by demographic fields.
- Visualized numeric distributions and group means to better understand the mental health survey data.

Further analysis can expand upon these steps to inform public health strategies and research in Kilifi County and beyond.